#### Import libraries

In [0]:

from pyspark.sql import DataFrame
from pyspark.sql import functions as SQL_FUNCTIONS

import matplotlib.pyplot as plt
import pandas as pd

#### Create useful variables and read the data table

In [0]:

dbutils.widgets.text("silver_table", "workspace.bda_taxi.taxi_silver")

silver_table = dbutils.widgets.get("silver_table").strip()
print("Silver table:", silver_table)

# Load Silver data

def read_table(table_fqn: str) -> DataFrame:
    return spark.table(table_fqn)

silver_dataframe = read_table(silver_table)
print("Silver rows:", silver_dataframe.count())
display(silver_dataframe.limit(5))

#### Helper functions for EDA: graphs etc.

In [0]:

def plot_histogram(
    dataframe: DataFrame,
    column_name: str,
    bins: int = 50,
    log_scale: bool = False,
    title_suffix: str = ""
) -> None:
    """
    Plots a histogram for a numeric column using Pandas + Matplotlib.
    Intended for distribution inspection and outlier analysis.
    """
    pdf = dataframe.select(column_name).dropna().toPandas()

    plt.figure(figsize=(8, 5))
    plt.hist(pdf[column_name], bins=bins)
    if log_scale:
        plt.yscale("log")
    plt.xlabel(column_name)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {column_name} {title_suffix}")
    plt.tight_layout()
    plt.show()


def show_quantiles(
    dataframe: DataFrame,
    column_name: str,
    quantiles: list = [0.5, 0.9, 0.95, 0.99, 0.995]
) -> None:
    """
    Displays approximate quantiles for a numeric column.
    Useful for justifying outlier thresholds.
    """
    values = dataframe.approxQuantile(column_name, quantiles, 0.01)
    for q, v in zip(quantiles, values):
        print(f"{column_name} quantile {q}: {v}")


def top_k_table_with_count(
    dataframe: DataFrame,
    group_column: str,
    metric_column: str,
    k: int = 10,
    metric_name: str = "avg_value",
    MIN_TRIPS: int = 100
) -> DataFrame:
    """
    Returns a top-k table by average metric, including trip count for context.
    """
    return (
        dataframe
        .groupBy(group_column)
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.avg(metric_column).alias(metric_name)
        )
        .filter(SQL_FUNCTIONS.col("trip_count") >= MIN_TRIPS)
        .orderBy(SQL_FUNCTIONS.desc(metric_name))
        .limit(k)
    )

#### Plot the charts and list out the data to allow examination

##### Distribution analysis and outlier justification

In [0]:
# Distribution analysis and outlier justification

plot_histogram(
    silver_dataframe,
    column_name="trip_distance",
    bins=10,
    log_scale=False,
    title_suffix="(linear scale)"
)

# Trip distance distribution
plot_histogram(
    silver_dataframe,
    column_name="trip_distance",
    bins=60,
    log_scale=True,
    title_suffix="(log-scaled)"
)

show_quantiles(silver_dataframe, "trip_distance")

# Fare amount distribution
plot_histogram(
    silver_dataframe,
    column_name="fare_amount",
    bins=60,
    log_scale=True,
    title_suffix="(log-scaled)"
)

show_quantiles(silver_dataframe, "fare_amount")

##### Temporal analysis

In [0]:
# Temporal analysis

# Trips by time bucket
trips_by_time_bucket = (
    silver_dataframe
    .groupBy("time_bucket")
    .count()
    .orderBy("time_bucket")
)

display(trips_by_time_bucket)


# Average fare by time bucket
avg_fare_by_time_bucket = (
    silver_dataframe
    .groupBy("time_bucket")
    .agg(SQL_FUNCTIONS.avg("fare_amount").alias("avg_fare"))
    .orderBy("time_bucket")
)

display(avg_fare_by_time_bucket)

##### Tipphing behaviour (credit card trips only)

In [0]:
# Tipping behaviour (credit card trips only)


# Restrict to trips where tips are recorded
tips_recorded_df = silver_dataframe.filter(SQL_FUNCTIONS.col("tip_recorded") == 1)

print("Trips with recorded tips:", tips_recorded_df.count())


# Tip rate by time bucket
tip_rate_by_time_bucket = (
    tips_recorded_df
    .groupBy("time_bucket")
    .agg(SQL_FUNCTIONS.count("*").alias("trip_count"), SQL_FUNCTIONS.avg("tipped").alias("tip_rate"))
    .orderBy("time_bucket")
)

display(tip_rate_by_time_bucket)


# Average tip amount by time bucket
avg_tip_by_time_bucket = (
    tips_recorded_df
    .groupBy("time_bucket")
    .agg(SQL_FUNCTIONS.count("*").alias("trip_count"), SQL_FUNCTIONS.avg("tip_amount").alias("avg_tip"))
    .orderBy("time_bucket")
)

display(avg_tip_by_time_bucket)

##### Spatial analysis (pickup and dropoff zones)

In [0]:
# Spatial analysis (pickup and dropoff zones)

# Top pickup zones by average fare
top_pickup_fare = top_k_table_with_count(
    silver_dataframe,
    group_column="PU_Zone",
    metric_column="fare_amount",
    k=10,
    metric_name="avg_fare"
)

display(top_pickup_fare)

# Top dropoff zones by average fare
top_dropoff_fare = top_k_table_with_count(
    silver_dataframe,
    group_column="DO_Zone",
    metric_column="fare_amount",
    k=10,
    metric_name="avg_fare"
)

display(top_dropoff_fare)

# Top pickup zones by average tip (credit card only)
top_pickup_tip = top_k_table_with_count(
    tips_recorded_df,
    group_column="PU_Zone",
    metric_column="tip_amount",
    k=10,
    metric_name="avg_tip"
)

display(top_pickup_tip)

# Top dropoff zones by average tip (credit card only)
top_dropoff_tip = top_k_table_with_count(
    tips_recorded_df,
    group_column="DO_Zone",
    metric_column="tip_amount",
    k=10,
    metric_name="avg_tip"
)

display(top_dropoff_tip)


#### [WIP]: Overlay on real map of NYC

In [0]:
import os
import json
import pandas as pd
import plotly.express as px

def load_geojson_by_location_id(geojson_path: str) -> dict:
    with open(geojson_path, "r") as f:
        geo = json.load(f)
    # map: LocationID (str) -> feature (polygon)
    return {str(feat["properties"]["locationid"]): feat for feat in geo["features"]}

def attach_geometry(avg_metric_pdf: pd.DataFrame, geo_by_id: dict, id_col: str) -> pd.DataFrame:
    df = avg_metric_pdf.copy()
    df[id_col] = df[id_col].astype(int).astype(str)
    df["geometry"] = df[id_col].apply(lambda x: geo_by_id.get(x))
    # drop rows with no geometry match
    return df.dropna(subset=["geometry"])

In [0]:
repo_root = os.getcwd()
geojson_path = os.path.join(repo_root, "data", "NYC_Taxi_Zones_20260206.geojson")

geo_by_id = load_geojson_by_location_id(geojson_path)

avg_tip_by_dropoff = (
    silver_dataframe
    .filter(SQL_FUNCTIONS.col("tip_recorded") == 1)
    .groupBy("PULocationID")
    .agg(SQL_FUNCTIONS.avg("tip_amount").alias("avg_tip"))
)

avg_tip_pdf = avg_tip_by_dropoff.toPandas()
avg_tip_pdf["PULocationID_str"] = avg_tip_pdf["PULocationID"].astype(int).astype(str)

zones_metric_pdf = attach_geometry(avg_tip_pdf.rename(columns={"PULocationID_str": "zone_id"}), geo_by_id, "zone_id")

print("Rows with geometry:", len(zones_metric_pdf))
zones_metric_pdf.head()

In [0]:
import os
import json
import pandas as pd
import plotly.express as px

# 1) Load GeoJSON (FeatureCollection)
repo_root = os.getcwd()
geojson_path = os.path.join(repo_root, "data", "NYC_Taxi_Zones_20260206.geojson")

with open(geojson_path, "r") as f:
    taxi_zones_geojson = json.load(f)

# 2) Aggregate metric
avg_tip_by_dropoff = (
    silver_dataframe
    .filter(SQL_FUNCTIONS.col("tip_recorded") == 1)
    .groupBy("PULocationID")
    .agg(SQL_FUNCTIONS.avg("tip_amount").alias("avg_tip"))
)

avg_tip_pdf = avg_tip_by_dropoff.toPandas()
avg_tip_pdf["PULocationID_str"] = avg_tip_pdf["PULocationID"].astype(int).astype(str)

# 3) Sanity check: do we have matching IDs?
geo_ids = set(str(f["properties"].get("LocationID")) for f in taxi_zones_geojson["features"])
df_ids = set(avg_tip_pdf["PULocationID_str"].unique())
matches = len(geo_ids.intersection(df_ids))

print("GeoJSON zones:", len(geo_ids))
print("Data zones:", len(df_ids))
print("Matches:", matches)

# 4) Plot choropleth with borders
fig = px.choropleth_mapbox(
    avg_tip_pdf,
    geojson=taxi_zones_geojson,
    locations="PULocationID_str",
    featureidkey="properties.locationid",
    color="avg_tip",
    mapbox_style="carto-positron",
    zoom=9.5,
    center={"lat": 40.73, "lon": -73.94},
    opacity=0.65,
    title="Average Tip Amount by Pick-up Taxi Zone (Credit Card Trips)"
)

# Make polygons visible even if colors are similar
fig.update_traces(marker_line_width=1.0)          # border thickness
fig.update_traces(marker_line_color="black")      # border color

fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
fig.show()